In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import random
import re
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
HOMOGLYPH_MAP = {
    'a': '\u0430',  # Cyrillic а
    'e': '\u0435',  # Cyrillic е
    'o': '\u043e',  # Cyrillic о
    'p': '\u0440',  # Cyrillic р
    'c': '\u0441',  # Cyrillic с
    'x': '\u0445',  # Cyrillic х
    'i': '\u0456',  # Cyrillic і (Ukrainian)
}

TOXIC_WORDS = {
    'kill', 'hate', 'die', 'murder', 'attack', 'destroy',
    'stupid', 'idiot', 'moron', 'trash', 'garbage', 'filth'
}

def insert_zero_width_spaces(word):
    result = []
    for i, ch in enumerate(word):
        result.append(ch)
        if (i + 1) % random.choice([2, 3]) == 0 and i < len(word) - 1:
            result.append('\u200b')
    return ''.join(result)

def apply_homoglyphs(word):
    return ''.join(HOMOGLYPH_MAP.get(ch.lower(), ch) for ch in word)

def duplicate_chars(word, prob=0.2):
    result = []
    for ch in word:
        result.append(ch)
        if random.random() < prob:
            result.append(ch)
    return ''.join(result)

def perturb(text):
    words = text.split()
    perturbed_words = []

    for word in words:
        word_lower = word.lower().strip('.,!?;:\'"')

        w = insert_zero_width_spaces(word)
        w = apply_homoglyphs(w)
        w = duplicate_chars(w, prob=0.2)

        perturbed_words.append(w)

    return ' '.join(perturbed_words)

sample_text = "I hate you and you should die"
print(f"Original:  {sample_text}")
print(f"Perturbed: {perturb(sample_text)}")

In [ ]:
df_eval = pd.read_csv('eval_subset.csv')
prob_toxic_clean = np.load('eval_probs_part1.npy')
true_labels = np.load('eval_labels.npy')

THRESHOLD = 0.5
pred_clean = (prob_toxic_clean >= THRESHOLD).astype(int)

high_conf_toxic_idx = np.where(
    (pred_clean == 1) & (prob_toxic_clean >= 0.7)
)[0]

np.random.seed(SEED)
if len(high_conf_toxic_idx) >= 500:
    sampled_idx = np.random.choice(high_conf_toxic_idx, size=500, replace=False)
else:
    sampled_idx = high_conf_toxic_idx
    print(f"WARNING: Only {len(high_conf_toxic_idx)} high-confidence toxic samples available")

attack_texts_original = df_eval.iloc[sampled_idx]['comment_text'].tolist()
attack_labels = true_labels[sampled_idx]
attack_probs_original = prob_toxic_clean[sampled_idx]

print(f"Attack sample size: {len(attack_texts_original)}")
print(f"Average confidence (before attack): {attack_probs_original.mean():.4f}")

In [ ]:
print("Applying perturbations...")
attack_texts_perturbed = [perturb(t) for t in attack_texts_original]
print("Done. Running perturbed texts through model...")

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = './model_checkpoint_part1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

def get_predictions(texts, batch_size=64):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, max_length=128, truncation=True,
            padding=True, return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

attack_probs_perturbed = get_predictions(attack_texts_perturbed)
attack_preds_perturbed = (attack_probs_perturbed >= THRESHOLD).astype(int)

asr = np.mean(attack_preds_perturbed == 0)

print("\n" + "=" * 50)
print("ATTACK 1: Character-Level Evasion Results")
print("=" * 50)
print(f"Sample size:                {len(attack_texts_original)}")
print(f"Avg confidence BEFORE:      {attack_probs_original.mean():.4f}")
print(f"Avg confidence AFTER:       {attack_probs_perturbed.mean():.4f}")
print(f"Attack Success Rate (ASR):  {asr:.4f} ({asr*100:.1f}%)")
print(f"  = fraction of originally-detected toxic comments now evading the classifier")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(attack_probs_original, bins=30, color='red', alpha=0.7, label='Before Attack')
axes[0].hist(attack_probs_perturbed, bins=30, color='blue', alpha=0.7, label='After Attack')
axes[0].axvline(THRESHOLD, color='black', linestyle='--', label='Threshold')
axes[0].set_xlabel('Predicted Toxicity Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution Before/After Evasion Attack')
axes[0].legend()

categories = ['Evaded (ASR)', 'Still Detected']
values = [asr, 1 - asr]
axes[1].bar(categories, values, color=['green', 'red'], alpha=0.8)
axes[1].set_ylabel('Fraction')
axes[1].set_title(f'Evasion Attack Outcome\n(ASR = {asr:.3f})')
axes[1].set_ylim(0, 1)
for i, v in enumerate(values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig('part3_evasion_attack.png', dpi=150)
plt.show()

In [ ]:

df_train = pd.read_csv('train_subset.csv')
print(f"Training set loaded: {len(df_train)} rows")

np.random.seed(SEED)
flip_idx = np.random.choice(len(df_train), size=int(0.05 * len(df_train)), replace=False)

df_poisoned = df_train.copy()
df_poisoned.loc[flip_idx, 'label'] = 1 - df_poisoned.loc[flip_idx, 'label']

n_flipped = (df_poisoned['label'] != df_train['label']).sum()
print(f"Labels flipped: {n_flipped} ({n_flipped/len(df_train)*100:.1f}%)")
print(f"Original toxic rate: {df_train['label'].mean():.4f}")
print(f"Poisoned toxic rate: {df_poisoned['label'].mean():.4f}")

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_length,
            truncation=True, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'accuracy': accuracy_score(labels, predictions)
    }

BASE_MODEL = 'distilbert-base-uncased'
tok_poisoned = AutoTokenizer.from_pretrained(BASE_MODEL)

df_eval_nb = pd.read_csv('eval_subset.csv')

poisoned_train_ds = ToxicityDataset(
    df_poisoned['comment_text'].tolist(),
    df_poisoned['label'].tolist(),
    tok_poisoned
)
clean_eval_ds = ToxicityDataset(
    df_eval_nb['comment_text'].tolist(),
    df_eval_nb['label'].tolist(),
    tok_poisoned
)

poisoned_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

poisoned_args = TrainingArguments(
    output_dir='./results_poisoned',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    evaluation_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

poisoned_trainer = Trainer(
    model=poisoned_model,
    args=poisoned_args,
    train_dataset=poisoned_train_ds,
    eval_dataset=clean_eval_ds,
    compute_metrics=compute_metrics,
)

print("Training poisoned model...")
poisoned_trainer.train()
print("Training complete!")

In [ ]:
poison_output = poisoned_trainer.predict(clean_eval_ds)
logits_p = poison_output.predictions
probs_p = F.softmax(torch.tensor(logits_p), dim=-1).numpy()[:, 1]
preds_p = (probs_p >= THRESHOLD).astype(int)

eval_labels_arr = df_eval_nb['label'].values

def get_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    return {
        'F1 Macro': f1_score(y_true, y_pred, average='macro'),
        'Accuracy': accuracy_score(y_true, y_pred),
        'FNR': fnr
    }

clean_metrics = get_metrics(eval_labels_arr, (prob_toxic_clean >= THRESHOLD).astype(int))
poison_metrics = get_metrics(eval_labels_arr, preds_p)

print("\n" + "=" * 55)
print("ATTACK 2: Label-Flipping Poisoning Results")
print("=" * 55)
comparison = pd.DataFrame({
    'Metric': ['F1 Macro', 'Accuracy', 'FNR (False Negative Rate)'],
    'Clean Model': [
        f"{clean_metrics['F1 Macro']:.4f}",
        f"{clean_metrics['Accuracy']:.4f}",
        f"{clean_metrics['FNR']:.4f}"
    ],
    'Poisoned Model': [
        f"{poison_metrics['F1 Macro']:.4f}",
        f"{poison_metrics['Accuracy']:.4f}",
        f"{poison_metrics['FNR']:.4f}"
    ],
    'Change': [
        f"{poison_metrics['F1 Macro'] - clean_metrics['F1 Macro']:+.4f}",
        f"{poison_metrics['Accuracy'] - clean_metrics['Accuracy']:+.4f}",
        f"{poison_metrics['FNR'] - clean_metrics['FNR']:+.4f}"
    ]
})
print(comparison.to_string(index=False))
print("\n(FNR increase = more toxic content now missed by the model)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics_names = ['F1 Macro', 'Accuracy', 'FNR']
clean_vals = [clean_metrics['F1 Macro'], clean_metrics['Accuracy'], clean_metrics['FNR']]
poison_vals = [poison_metrics['F1 Macro'], poison_metrics['Accuracy'], poison_metrics['FNR']]

x = np.arange(len(metrics_names))
width = 0.35
ax.bar(x - width/2, clean_vals, width, label='Clean Model', color='#2ecc71', alpha=0.8)
ax.bar(x + width/2, poison_vals, width, label='Poisoned Model (5% flipped)', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Clean vs Poisoned Model Performance')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('part3_poisoning_attack.png', dpi=150)
plt.show()

## 3.3 Operational Risk Analysis

### Which attack is more operationally dangerous?

**The evasion attack is more immediately operationally dangerous for a live platform**, but the poisoning attack represents a more systemic, harder-to-detect long-term threat.

**Evasion attack analysis:**
- Requires per-comment effort from the attacker (applying character substitutions)
- Can be deployed at scale with a simple script — a bot farm can perturb thousands of comments per second
- The effort is trivially automatable, making the 'per-comment' cost negligible in practice
- The ASR reveals that a non-trivial fraction of originally-detected toxic comments evade detection after perturbation
- **Detection**: the zero-width spaces and Cyrillic characters are detectable with simple Unicode normalization as a pre-filter

**Poisoning attack analysis:**
- Requires write access to the training pipeline — either through compromised data collection, crowdworker manipulation, or insider access
- On a social platform, **the training data IS user-generated content**, meaning any user can contribute poisoned examples to future training rounds if the platform uses active learning or continuous retraining
- The FNR increase means more genuine toxicity goes undetected, which is the attacker's goal in many threat models
- Poisoning is much harder to detect than evasion — it leaves no visible artifact in the content itself

**Threat model assessment:**

The evasion attack is the more **realistic near-term threat** because it requires only content-level access (which every user has). The poisoning attack is the more **catastrophic long-term threat** because it corrupts the model itself. Defenses should be prioritized in this order:
1. **Immediate**: Unicode normalization and character canonicalization as a pre-filter (neutralizes most evasion attacks cheaply)
2. **Medium-term**: Data validation and anomaly detection in the training pipeline to flag unusual label distributions before retraining
3. **Long-term**: Certified defenses against poisoning (e.g., data sanitization, differential privacy during training)